# Оценка кризисных детекторов

|  |  |
|---|---|
| Данные | 98 синтетических реплик из `crisis_holdout.yaml` |
| Варианты | Словарь, классификатор, оба детектора вместе |
| Цель | Найти хотя бы 95 % явных кризисных реплик |

Словарь ищет знакомую формулировку и учитывает, о ком идёт речь. Классификатор оценивает смысл текста и может найти сигнал, для которого нет шаблона. В чате достаточно кризисной находки любого из них. Сравним число пропусков и ложных тревог у каждого детектора и при совместной проверке.

## 1. Контрольная выборка

Для явных и косвенных кризисных реплик (`explicit` и `implicit`) ожидаем переход в кризисный режим. В остальных группах считаем такой переход ложной тревогой. Пассивные мысли относятся к отдельной категории и позволяют продолжить диалог. Контрольный набор составлен до настройки словаря и обучения классификатора. Старый набор `crisis_replies.yaml` использовался для подбора шаблонов, поэтому в контрольную оценку входят только реплики из `crisis_holdout.yaml`.

In [1]:
from pathlib import Path
import sys
from typing import Final, cast

import numpy as np
import pandas as pd

from anamnesis.application.interfaces.safety.crisis_detector import ICrisisDetector
from anamnesis.domain.enums.crisis import CrisisCategory
from anamnesis.domain.types import LanguageCode
from anamnesis.infrastructure.filesystem.yaml_reader import read_yaml
from anamnesis.infrastructure.safety.combined_detector import CombinedCrisisDetector
from anamnesis.infrastructure.safety.encoder_detector import ClassifierFile, EncoderCrisisDetector
from anamnesis.infrastructure.safety.lexicon_detector import LexiconCrisisDetector
from anamnesis.infrastructure.text.language import LanguageProfiles
from anamnesis.main.config.config import Config
from anamnesis.main.config.knowledge import KnowledgeConfig


if Path.cwd().name == "benchmarks":
    sys.path.insert(0, str(Path.cwd().parent))

from benchmarks.utils.support import load_data, use_project_root


use_project_root()

ru: Final[LanguageCode] = LanguageCode("ru")
config: KnowledgeConfig = Config.load().knowledge
parameters: ClassifierFile = read_yaml(
    config.resources_dir / ru / "crisis_classifier.yaml",
    ClassifierFile,
)
profiles: LanguageProfiles = LanguageProfiles.load(config.resources_dir, (ru,))
lexicon: LexiconCrisisDetector = LexiconCrisisDetector.load(config.resources_dir, profiles)
encoder: EncoderCrisisDetector = EncoderCrisisDetector.load(
    config.resources_dir,
    (ru,),
    config.models_dir,
)
detectors: dict[str, ICrisisDetector] = {
    "Словарь": lexicon,
    "Классификатор": encoder,
    "Вместе": CombinedCrisisDetector(lexicon, encoder),
}
groups: dict[str, list[str]] = cast(
    dict[str, list[str]],
    load_data("crisis_holdout.yaml"),
)
print(f"Энкодер {parameters.encoder}, порог {parameters.threshold:.6f}")

Энкодер rubert-tiny2, порог 0.726283


In [2]:
rows: list[dict[str, str | bool]] = []
group: str
texts: list[str]
text: str
name: str
detector: ICrisisDetector
for group, texts in groups.items():
    for text in texts:
        for name, detector in detectors.items():
            categories: set[CrisisCategory] = {
                signal.category for signal in await detector.detect(text, ru)
            }
            crisis: bool = any(category.is_crisis for category in categories)
            passive_only: bool = CrisisCategory.PASSIVE in categories and not crisis
            rows.append(
                {
                    "Детектор": name,
                    "Группа": group,
                    "Реплика": text,
                    "Кризис": crisis,
                    "Пассивный сигнал без кризиса": passive_only,
                }
            )

results: pd.DataFrame = pd.DataFrame(rows)
counts: pd.DataFrame = pd.DataFrame(
    {
        "Реплик": {group: len(texts) for group, texts in groups.items()},
    }
)
transitions: pd.DataFrame = results.pivot_table(
    index="Группа",
    columns="Детектор",
    values="Кризис",
    aggfunc="sum",
)
counts.join(transitions)[["Реплик", *detectors]]

,Реплик,Словарь,Классификатор,Вместе
explicit,30,14,10,18
implicit,25,0,5,5
passive,8,0,0,0
negation,10,2,0,2
quote,8,5,0,5
other_person,5,0,0,0
neutral,12,1,0,1


## 2. Метрики

В таблице выше указано число переходов по каждой группе. Полноту считаем отдельно для явных и косвенных кризисных реплик. Точность показывает, какая доля переходов пришлась на эти две группы. В последнем столбце считаем реплики, для которых детектор распознал пассивные мысли и диалог продолжился.

In [3]:
type MetricValue = str | int | float | np.int64 | np.float64

metrics: list[dict[str, MetricValue]] = []
part: pd.DataFrame
for name, part in results.groupby("Детектор", sort=False):
    explicit: pd.Series = part["Группа"].eq("explicit")
    implicit: pd.Series = part["Группа"].eq("implicit")
    passive: pd.Series = part["Группа"].eq("passive")
    expected: pd.Series = explicit | implicit
    predicted: pd.Series = part["Кризис"]
    correct: int | np.int64 = (expected & predicted).sum()

    metrics.append(
        {
            "Детектор": name,
            "Полнота, явные": predicted[explicit].mean(),
            "Полнота, косвенные": predicted[implicit].mean(),
            "Точность": correct / predicted.sum() if predicted.any() else float("nan"),
            "Пропуски": (expected & ~predicted).sum(),
            "Ложные тревоги": (~expected & predicted).sum(),
            "Пассивные распознаны": part.loc[passive, "Пассивный сигнал без кризиса"].sum(),
        }
    )

pd.DataFrame(metrics).set_index("Детектор").round(3)

,"Полнота, явные","Полнота, косвенные",Точность,Пропуски,Ложные тревоги,Пассивные распознаны
Детектор,,,,,,
Словарь,0.467,0.0,0.636,41,8,1
Классификатор,0.333,0.2,1.000,40,0,0
Вместе,0.600,0.2,0.742,32,8,1


## Выводы

1. Словарь обнаружил 14 из 30 явных кризисных реплик. Классификатор добавил ещё четыре явные и пять косвенных. Вместе детекторы нашли 18 из 30 явных (60 %) при цели 95 % и 5 из 25 косвенных (20 %).
2. Все восемь ложных тревог дал словарь. Пять пришлись на цитаты и обсуждения, две — на отрицания и одна — на нейтральную реплику. В жалобе на принтер словарь сработал на слово «застрелился». У классификатора на этом наборе ложных тревог не было.
3. Ни одна из восьми пассивных реплик не перевела чат в кризисный режим. Только в одной словарь распознал пассивные мысли. Отсутствие ложного перехода и способность заметить такой сигнал здесь дают разные результаты.